---
title: "Chapter 6: Errors and Limitations"
short_title: "6. Errors and Limitations"
---

This chapter summarizes the automatic checks of `validate.py` and the cases that were not crawled completely. The check levels have the following meaning:

| Level | Meaning |
| :--- | :--- |
| `PASS` | The check passed |
| `WARN` | Requires an explanation or a manual review, for example a missing arXiv date of a reference, or a 404 version already checked by hand |
| `PENDING` | Unfinished work, for example a `failed` version awaiting a rerun, or a paper without a reference file after Semantic Scholar answered 429 |
| `FAIL` | Structural or content error: missing file, empty file, inconsistent reference data |

In [ ]:
import pandas as pd
import plotly.express as px
from report_utils import (COLORS, FINAL_SOURCE, REF_OK, group, human_bytes, human_seconds,
                          load, members, meta, metric, mib, pct, provenance, show, table)
provenance()

## 1. Automatic check results

In [ ]:
vc = load("validation_counts")
pivot = vc.pivot_table(index="check_name", columns="status", values="count", aggfunc="sum", fill_value=0)
pivot = pivot.reindex(columns=[c for c in ["PASS", "WARN", "PENDING", "FAIL"] if c in pivot.columns])
pivot.columns.name = None
table(pivot.reset_index().rename(columns={"check_name": "Check"}), "Table 6.1. Results per check and level")
bad = vc[vc["status"] != "PASS"].groupby(["check_name", "status"])["count"].sum().reset_index()
if len(bad):
    fig = px.bar(bad, x="count", y="check_name", color="status", orientation="h",
                 color_discrete_map={"WARN": "#DDAA33", "PENDING": "#4477AA", "FAIL": "#BB5566"},
                 title="Checks not passed (PASS excluded)",
                 labels={"count": "Results", "check_name": "Check", "status": "Level"})
    show(fig)
else:
    print("Every check passed.")

## 2. Incompletely crawled cases

In [ ]:
papers, versions, m = load("papers"), load("versions"), members()
found = papers[papers["discovery_status"] == "success"]
cases = [
    ("Discovery", "IDs not found on arXiv (not_found)", int(m["ids_not_found"].sum())),
    ("Discovery", "IDs with remaining API errors (api_error)", int(m["ids_api_error"].sum())),
    ("Source", "Versions answering HTTP 404 (not_found): withdrawn, see Section 3", int((versions["status"] == "not_found").sum())),
    ("Source", "PDF-only versions (pdf_only)", int((versions["status"] == "pdf_only").sum())),
    ("Source", "Versions without .tex/.bib files (no_tex)", int((versions["status"] == "no_tex").sum())),
    ("Source", "Failed versions: no TeX source, see Section 3", int((versions["status"] == "failed").sum())),
    ("Source", "Versions without a status (missing)", int((versions["status"] == "missing").sum())),
    ("References", "Papers not indexed by Semantic Scholar (not_found)", int((found["ref_status"] == "not_found").sum())),
    ("References", "Papers whose references are not shared (unavailable)", int((found["ref_status"] == "unavailable").sum())),
    ("References", "Papers without a reference file (API error in the last run)", int((found["ref_status"] == "missing").sum())),
    ("References", "Papers missing arXiv dates of their references",
     int(found["arxiv_dates_complete"].astype(str).eq("False").sum())),
]
table(pd.DataFrame(cases, columns=["Stage", "Case", "Count"]), "Table 6.2. Incomplete cases, whole group")

## 3. Manual checks of versions without a source

An HTTP 404 does not reveal whether a version was withdrawn or whether arXiv failed temporarily, so `validate.py` marks every 404 version as `PENDING` until it has been reviewed (option `--verified-404`); `failed` versions are `PENDING` as well. Every such version was reviewed on its arXiv abstract page `arxiv.org/abs/<id>v<N>`, reading only the abstract page and never downloading the source again. Each member reviewed the 404 versions of their own range, after which a script read the *Submission history* of the complete list again. The `failed` versions were reviewed separately, from their abstract pages and the recorded download errors.

- **404 versions:** when the authors withdraw a paper, arXiv creates a new version without a source. The *Submission history* marks that version as `(withdrawn)`, and the *Access Paper* box shows *Withdrawn*.
- **`failed` versions:** arXiv answers HTTP 200 with a file of the same size on every attempt. Three versions are HTML submissions (proceedings or collections of conference contributions), delivered as a gzipped HTML page, which `extractor.py` takes for a server error page. The fourth is a PDF whose `%PDF` header follows two spaces, so its format is not recognized. All four are therefore classified as `failed` and retried in every pass, including the late sweeps ([Chapter 5](./05_performance.ipynb), Section 6), although they should be classified as `no_tex` and `pdf_only`.

The results are stored in `data/m1/manual_checks.csv`. The report data was exported before the manual checks, so Table 6.1 still lists these versions as `PENDING`; rerunning `validate.py --verified-404` with the checked list turns the 404 versions into `WARN`.

In [ ]:
checks = load("manual_checks")
key = ["member", "arxiv_id", "version"]
matched = checks.merge(versions[key + ["status", "attempts", "downloaded_bytes"]], on=key, how="left")
assert (matched["status"] == matched["crawl_status"]).all(), "manual_checks.csv does not match versions.csv"
rows = []
for name in m["member"]:
    v = versions[versions["member"] == name]
    c = matched[matched["member"] == name]
    rows.append({"Member": name,
                 "404 versions": int((v["status"] == "not_found").sum()),
                 "404 reviewed": int((c["crawl_status"] == "not_found").sum()),
                 "Withdrawn": int((c["finding"] == "withdrawn").sum()),
                 "Failed versions": int((v["status"] == "failed").sum()),
                 "Failed reviewed": int((c["crawl_status"] == "failed").sum()),
                 "HTML submission": int((c["finding"] == "html_source").sum()),
                 "PDF": int((c["finding"] == "pdf_source").sum())})
view = pd.DataFrame(rows)
view.loc[len(view)] = ["Group", *view.drop(columns="Member").sum().tolist()]
table(view, "Table 6.3. Results of the review on the arXiv abstract pages")
left = matched[matched["crawl_status"] == "failed"]
table(pd.DataFrame({"Member": left["member"], "Version": left["arxiv_id"] + left["version"],
                    "Download attempts": left["attempts"], "Downloaded size": left["downloaded_bytes"].map(human_bytes),
                    "Evidence": left["evidence"]}),
      "Table 6.4. Failed versions: none has a TeX source")
pending = int(vc.loc[(vc["check_name"] == "version") & (vc["status"] == "PENDING"), "count"].sum())
print(f"{len(matched):,} versions reviewed, against {pending:,} PENDING results of the version check in Table 6.1:")
print(f"  {(matched['finding'] == 'withdrawn').sum():,} withdrawn versions, "
      f"{(matched['finding'] == 'html_source').sum():,} HTML submissions, {(matched['finding'] == 'pdf_source').sum():,} PDF.")
print("None of them is a transient arXiv error; the failed versions are misclassified by extractor.py.")

## 4. Examples of results not passed

At most 30 rows are shown, `FAIL` first, then `PENDING` and `WARN` (the complete list is in the `validation_report.csv` file of each member).

In [ ]:
issues = load("validation_issues")
order = {"FAIL": 0, "PENDING": 1, "WARN": 2}
if len(issues):
    issues = issues.assign(o=issues["status"].map(order)).sort_values(["o", "member", "arxiv_id"]).drop(columns="o")
    table(issues.head(30).fillna(""), "Table 6.5. Examples of results not passed")
else:
    print("No result other than PASS.")
w = meta()["warnings"]
if w:
    print("Warnings raised when exporting the report data:")
    for line in w:
        print(" -", line)

## 5. Causes and handling

| Issue | Cause | Handling |
| :--- | :--- | :--- |
| arXiv answers HTTP 429/503 under heavy download | arXiv's download limits, which apply to all machines as a whole; according to arXiv, a 429 reflects the overall load of the server rather than the activity of one user | Only the refused route pauses while the other routes continue; when no route is left, every request pauses with increasing waits; a file that keeps failing is deferred to a later pass ([Chapter 2](./02_methodology.ipynb), Section 7) |
| A version answers 404 | Withdrawn version: arXiv keeps no source for it | Status `not_found`, no retry; **every** case reviewed on `arxiv.org/abs/<id><version>` (Section 3) |
| `failed` version without a TeX source | HTML submission taken for an error page, or PDF with blanks before its header | Reviewed (Section 3). Recognizing both cases in `extractor.py` (`no_tex`, `pdf_only`) would remove these unnecessary retries |
| PDF-only or DOCX version, or an archive without `.tex`/`.bib` files | The authors submitted no TeX source | Empty folder with status `pdf_only`/`no_tex`; the paper is kept |
| Semantic Scholar answers 429 | API rate limit | Dedicated API key, several papers per `/paper/batch` call, retries with waits; a failed paper is never recorded as having no references |
| Recent papers missing from Semantic Scholar | Semantic Scholar indexes papers a few days after they appear on arXiv | Status `not_found`; `references.py --retry not_found` can be run later |
| Missing arXiv dates of references | One extra arXiv API call per 100 cited IDs | Dates taken from the arXiv metadata snapshot on Kaggle (`ref_dates_kaggle.py`), at no cost in arXiv requests |
| Colab disconnects midway | Runtime time limit | Per-version checkpoints and a Drive snapshot after each step; a new runtime restores the data and resumes from the unfinished step. An interrupted step is excluded from the wall time and measured separately ([Chapter 5](./05_performance.ipynb), Section 6) |

## 6. Limitations

- **Running time:** because of Colab's 12-hour limit, the time of the group is the sum of the steps across several sessions rather than one continuous run. Snapshot saving and restoring, setup and idle time between sessions are not included in the wall time; an interrupted step is excluded (it is listed in the warnings) and rerun in the next session.
- **Sampled RAM and disk measurements:** RAM peaks shorter than 2 s, or disk peaks between two 300 s scans, may be missed. The measurements cover only the data folders (including the `.bbl` files): the temporary archives created for snapshots and the Kaggle metadata file (in the `kagglehub` cache) also reside on the runtime's disk but are not counted.
- **References depend on Semantic Scholar:** the coverage and quality of the reference metadata are those of Semantic Scholar; the group does not fill in information it cannot verify.
- **ID list:** the assigned IDs are taken from the list of every ID in each range (`assigned_ids.py`; for a range spanning several months, the end of each month is taken from the arXiv metadata on Kaggle rather than from the crawl results), and `validate.py --assigned-ids` checks each ID against the data. When the exported data lacks this list, the assigned IDs are the IDs of the manifests.